# 🏛️ NHS360 – Huấn Luyện Mô Hình Nhận Diện Địa Danh Ngũ Hành Sơn
### Khóa Luận Tốt Nghiệp SE13 – Smart Travel Ngũ Hành Sơn (FPT University Da Nang)

Notebook này được thiết kế để chạy trực tiếp trên **Google Colab miễn phí** với GPU (Tesla T4).
- **Mục tiêu**: Nhận diện 10 địa danh tiêu biểu tại Ngũ Hành Sơn mà **không cần dùng Cloud Vision / Gemini API bên thứ 3**.
- **Kiến trúc**: Transfer Learning trên nền **MobileNetV3-Small** (nhẹ, tối ưu cho thiết bị di động).
- **Đầu ra**: File `ngu_hanh_son_model.tflite` và `labels.txt` để nhúng trực tiếp vào ứng dụng Flutter di động.

## 1. Kiểm tra phần cứng GPU và phiên bản TensorFlow
*(Hãy chắc chắn bạn đã vào menu: **Runtime -> Change runtime type -> Hardware accelerator: GPU (T4)**)*

In [ ]:
!nvidia-smi
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

## 2. Chuẩn bị tập dữ liệu ảnh (Dataset)
Bạn có 2 lựa chọn:
- **Cách A**: Nén thư mục ảnh `dataset.zip` và upload trực tiếp lên Colab.
- **Cách B**: Kết nối Google Drive nếu đã lưu ảnh trên Drive.
- **Cách C**: Dùng công cụ sinh dữ liệu mẫu giả lập (Synthetic data) để test toàn bộ pipeline trước.

In [ ]:
# Tải mã nguồn huấn luyện nếu chưa có
!git clone https://github.com/vothuyen/NHS360_Capstone.git /content/repo || true
%cd /content

# Hoặc tạo thư mục làm việc trực tiếp
!mkdir -p dataset exported_model

In [ ]:
# Nếu bạn đã có file dataset.zip tải lên Colab, hãy giải nén:
# !unzip -q dataset.zip -d dataset/

# Hoặc sinh tập ảnh mẫu để kiểm thử quy trình ngay:
import os
from PIL import Image, ImageDraw

classes = [
    "dong_huyen_khong", "chua_tam_thai", "chua_linh_ung", "dong_van_thong",
    "dong_am_phu", "lang_da_non_nuoc", "vong_giang_dai", "dinh_thuong_thai",
    "chua_quan_the_am", "bai_bien_son_thuy"
]

for idx, c in enumerate(classes):
    folder = os.path.join("dataset", c)
    os.makedirs(folder, exist_ok=True)
    # Tạo 25 ảnh mẫu mỗi lớp nếu thư mục đang trống
    if len(os.listdir(folder)) < 5:
        for i in range(25):
            img = Image.new("RGB", (224, 224), color=((idx * 25) % 255, (idx * 50) % 255, (idx * 75) % 255))
            d = ImageDraw.Draw(img)
            d.rectangle([20, 20, 204, 204], outline=(255, 255, 255), width=2)
            d.text((30, 30), c[:15], fill=(255, 255, 255))
            img.save(os.path.join(folder, f"{c}_{i:03d}.jpg"))

print("✅ Chuẩn bị dataset hoàn tất với 10 lớp địa danh!")

## 3. Nạp dữ liệu và cấu hình Data Augmentation
Data Augmentation giúp mô hình học các biến thể góc chụp, ánh sáng và phóng to/thu nhỏ, tránh hiện tượng overfitting khi số lượng ảnh còn ít.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
print("Danh sách nhãn địa danh:", class_names)

# Lưu danh sách nhãn ra labels.txt
with open("exported_model/labels.txt", "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")

# Tối ưu bộ nhớ cache và prefetch
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

## 4. Xây dựng mô hình Transfer Learning (MobileNetV3-Small)

In [ ]:
# Base model pre-trained trên tập ImageNet
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)
x = base_model(x, training=False)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = keras.Model(inputs, outputs, name="NHS360_Landmark_Classifier")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)
model.summary()

## 5. Tiến hành Huấn Luyện (Training)

In [ ]:
EPOCHS = 20
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 6. Đồ thị độ chính xác (Accuracy & Loss)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.title('Độ chính xác qua các Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.title('Hàm mất mát (Loss)')
plt.legend()
plt.show()

## 7. Chuyển đổi và Xuất mô hình sang TensorFlow Lite (.tflite)

In [ ]:
# Xuất file TFLite chuẩn
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = "exported_model/ngu_hanh_son_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

import os
size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"✅ Đã xuất mô hình TFLite thành công: {tflite_path} ({size_mb:.2f} MB)")

## 8. Tải file mô hình về máy để tích hợp vào ứng dụng Flutter

In [ ]:
from google.colab import files

print("📥 Đang tải file mô hình...")
files.download("exported_model/ngu_hanh_son_model.tflite")
files.download("exported_model/labels.txt")

print("\n👉 Hướng dẫn: Copy 2 file này vào thư mục: Mobile/assets/models/ trong dự án NHS360!")